In [50]:
# Import dependencies
import numpy as np
import pandas as pd

In [51]:
# Load in the dataset
df = pd.read_csv('../../datasets/heart.csv')

X = df.drop(columns=['target']).to_numpy()
y = df['target'].to_numpy().reshape((303,1)) # Force the 1 dimension

# Perform Z-score normalization on x
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)

In [52]:
# Split into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, shuffle=True)

In [53]:
# Binary Cross Entropy (Log Loss) implementation
def BCELoss(y: np.ndarray, y_hat: np.ndarray):
    # Clip y_hat to be between epsilon and 1-epsilon
    # This prevents log(0)
    y_hat = np.clip(y_hat, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

In [ ]:
# Hyperparameters
learning_rate = 0.0005
epochs = 1750

In [55]:
class LogisticRegression:
    def __init__(self, num_features, learning_rate):
        self.lr = learning_rate
        self.w = np.zeros((num_features, 1))
        self.b = 0

        # For adam gradient descent
        self.w_m = 0
        self.w_v = 0
        self.b_m = 0
        self.b_v = 0
        self.beta1 = 0.9
        self.beta2 = 0.999

    def forward(self, X: np.ndarray):
        # Get linear component (z) and add a quadratic component (feature scaling)
        z = (X @ self.w) + self.b

        # Apply sigmoid
        return 1 / (1 + np.exp(-z))

    def update_weights(self, X: np.ndarray, y: np.ndarray, iteration: int):
        n = len(y) # Get length of examples

        # Get predictions
        y_hat = self.forward(X)

        # Perform ADAM gd for each feature
        dw = (1/n) * (X.T @ (y_hat - y))
        db = (1/n) * np.sum(y_hat - y)

        # Adam momentums
        self.w_m = self.beta1 * self.w_m + (1 - self.beta1) * dw # SGD + Momentum
        self.w_v = self.beta2 * self.w_v + (1 - self.beta2) * dw * dw # RMS Prop

        self.b_m = self.beta1 * self.b_m + (1 - self.beta1) * db # SGD + Momentum
        self.b_v = self.beta2 * self.b_v + (1 - self.beta2) * db * db # RMS Prop

        # Unbias them
        self.w_m_hat = self.w_m / (1 - self.beta1 ** iteration)
        self.w_v_hat = self.w_v / (1 - self.beta2 ** iteration)

        self.b_m_hat = self.b_m / (1 - self.beta1 ** iteration)
        self.b_v_hat = self.b_v / (1 - self.beta2 ** iteration)

        # Apply GD
        self.w -= (self.lr * self.w_m_hat) / (np.sqrt(self.w_v_hat) + 1e-7)
        self.b -= (self.lr * self.b_m_hat) / (np.sqrt(self.b_v_hat) + 1e-7)

    # Fit the data onto the logistic regression function
    def fit(self, X: np.ndarray, y: np.ndarray, epochs=1000, verbose=False):
        for i in range(epochs):
            # Perform gradient descent
            self.update_weights(X, y, i + 1) # plus 1 to avoid divide by zero

            # Report loss if verbose
            if verbose and i % 100 == 0:
                loss = BCELoss(y, self.forward(X))
                print(f'[{i}]: Loss: {loss:.4f}')


In [56]:
model = LogisticRegression(X.shape[1], learning_rate=learning_rate)
model.fit(X_train, y_train, epochs, True)

[0]: Loss: 0.6922
[100]: Loss: 0.6116
[200]: Loss: 0.5532
[300]: Loss: 0.5103
[400]: Loss: 0.4781
[500]: Loss: 0.4533
[600]: Loss: 0.4340
[700]: Loss: 0.4185
[800]: Loss: 0.4059
[900]: Loss: 0.3956
[1000]: Loss: 0.3870
[1100]: Loss: 0.3798
[1200]: Loss: 0.3737
[1300]: Loss: 0.3685
[1400]: Loss: 0.3640


In [57]:
# Evaluate model
def eval(X: np.ndarray, y: np.ndarray, model: LogisticRegression):
    # Get predictions (from actual labels)
    y_hat = model.forward(X)
    y_hat = (y_hat >= 0.5).astype(int)

    # Calculate accruacy
    accuracy = np.mean(y_hat == y)
    print(f'Test accuracy: {accuracy:.3f}')

eval(X_test, y_test, model)

Test accuracy: 0.848
